In [ ]:
# --- Paso 1: Importaciones ---
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, accuracy_score
from tensorflow import keras
from tensorflow.keras import layers

# Semilla fija para que la red empiece siempre con los mismos pesos (resultados reproducibles)
keras.utils.set_random_seed(42)

# --- Paso 2: Cargar el dataset limpio ---
df = pd.read_csv('safety_save_limpio.csv')

# Columnas del dataset limpio: texto libre de la observación y etiqueta de riesgo
X_texto = df['texto']       # columna con el texto libre
y = df['riesgo']          # columna con la etiqueta alto/bajo

# Convertir la etiqueta a formato numérico (alto=1, bajo=0)
y = y.map({'alto': 1, 'bajo': 0}).values

# --- Paso 3: Vectorización TF-IDF (min_df=2 y palabras de 3+ letras, decididos en el análisis exploratorio) ---
vectorizer = TfidfVectorizer(
    min_df=2,
    token_pattern=r'\b[a-zA-Z]{3,}\b',   # solo palabras de 3+ letras
    stop_words=None                       # sin lista de stopwords, para conservar "not"
)
X = vectorizer.fit_transform(X_texto).toarray()

print(f"Tamaño del vocabulario: {X.shape[1]} palabras")
print(f"Número de observaciones: {X.shape[0]}")

# --- Paso 4: Función para construir un modelo nuevo en cada ronda ---
def construir_modelo(input_dim):
    modelo = keras.Sequential([
        layers.Dense(8, activation='relu', input_dim=input_dim),
        layers.Dense(1, activation='sigmoid')
    ])
    modelo.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return modelo

# --- Paso 5: Validación cruzada estratificada de 5 particiones ---
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

exactitudes = []
predicciones_totales = []
etiquetas_reales_totales = []

for ronda, (idx_train, idx_test) in enumerate(skf.split(X, y), start=1):
    X_train, X_test = X[idx_train], X[idx_test]
    y_train, y_test = y[idx_train], y[idx_test]

    modelo = construir_modelo(input_dim=X.shape[1])
    modelo.fit(X_train, y_train, epochs=50, batch_size=8, verbose=0)

    y_pred_prob = modelo.predict(X_test, verbose=0)
    y_pred = (y_pred_prob > 0.5).astype(int).flatten()

    acc = accuracy_score(y_test, y_pred)
    exactitudes.append(acc)

    predicciones_totales.extend(y_pred)
    etiquetas_reales_totales.extend(y_test)

    print(f"Ronda {ronda}: exactitud = {acc:.3f}")

# --- Paso 6: Resultados finales ---
print(f"\nExactitud promedio (5 rondas): {np.mean(exactitudes):.3f}")
print(f"Desviación estándar: {np.std(exactitudes):.3f}")

matriz = confusion_matrix(etiquetas_reales_totales, predicciones_totales)
print("\nMatriz de confusión acumulada (5 rondas):")
print(matriz)

Tamaño del vocabulario: 79 palabras
Número de observaciones: 105


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Ronda 1: exactitud = 0.619


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Ronda 2: exactitud = 0.810


...

Ronda 3: exactitud = 0.905


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Ronda 4: exactitud = 0.667


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Ronda 5: exactitud = 0.571

Exactitud promedio (5 rondas): 0.714
Desviación estándar: 0.124

Matriz de confusión acumulada (5 rondas):
[[36 19]
 [11 39]]


In [ ]:
# --- Paso 7: Comparación de umbrales de decisión ---
# Repite la misma validación cruzada con la misma semilla, pero guarda las
# probabilidades de cada observación para probar varios umbrales de decisión.
keras.utils.set_random_seed(42)

probabilidades = np.zeros(len(y))

for idx_train, idx_test in skf.split(X, y):
    modelo = construir_modelo(input_dim=X.shape[1])
    modelo.fit(X[idx_train], y[idx_train], epochs=50, batch_size=8, verbose=0)
    probabilidades[idx_test] = modelo.predict(X[idx_test], verbose=0).flatten()

print("Umbral | Exactitud | Altos detectados | Altos no detectados | Falsas alarmas")
for umbral in [0.3, 0.4, 0.5, 0.6]:
    y_pred = (probabilidades > umbral).astype(int)
    verdaderos_bajos, falsas_alarmas, no_detectados, detectados = confusion_matrix(y, y_pred).ravel()
    exactitud = accuracy_score(y, y_pred)
    print(f"  {umbral:.1f}  |   {exactitud:.3f}   |     {detectados} de 50     |        {no_detectados}           |       {falsas_alarmas}")

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Umbral | Exactitud | Altos detectados | Altos no detectados | Falsas alarmas
  0.3  |   0.648   |     44 de 50     |        6           |       31
  0.4  |   0.743   |     44 de 50     |        6           |       21
  0.5  |   0.714   |     39 de 50     |        11           |       19
  0.6  |   0.638   |     25 de 50     |        25           |       13


## Resultados oficiales (semilla 42, reproducibles)

- Exactitud promedio (5 rondas): 71.4%
- Desviación estándar: 0.124
- Matriz de confusión acumulada con umbral 0.5: 36 bajos correctos, 19 falsas alarmas, 11 altos no detectados, 39 altos detectados

## Comparación de umbrales de decisión

| Umbral | Exactitud | Altos detectados | Altos no detectados | Falsas alarmas |
|---|---|---|---|---|
| 0.3 | 64.8% | 44 de 50 | 6 | 31 |
| 0.4 | 74.3% | 44 de 50 | 6 | 21 |
| 0.5 | 71.4% | 39 de 50 | 11 | 19 |
| 0.6 | 63.8% | 25 de 50 | 25 | 13 |

## Decisión: umbral 0.4

El umbral de 0.4, si bien genera un número significativo de falsas alarmas (21 de 55 observaciones de riesgo bajo), se justifica en un contexto de seguridad industrial, porque es preferible una alerta de más a que condiciones de alto riesgo se clasifiquen como de bajo riesgo. La diferencia entre el umbral de 0.5 y el de 0.4 reduce los riesgos no detectados de 11 a 6, a cambio de solo 2 falsas alarmas más.

Estos resultados deben interpretarse con cautela por dos razones: la muestra es limitada (105 observaciones) y el umbral se eligió con los mismos datos con los que se evalúa el modelo, lo que puede hacer que su desempeño se vea más favorable de lo que sería con observaciones nuevas.

In [ ]:
# --- Paso 8: Riesgo por contratista a partir de las probabilidades del perceptrón ---
UMBRAL = 0.4
df['probabilidad_alto'] = probabilidades

resumen = df.groupby('contratista').agg(
    observaciones=('probabilidad_alto', 'size'),
    riesgo_promedio=('probabilidad_alto', 'mean'),
    porcentaje_sobre_umbral=('probabilidad_alto', lambda p: (p > UMBRAL).mean() * 100)
).round(3)

# Según la sección C.1, solo A, B y C tienen muestra suficiente;
# los demás reciben el riesgo promedio de todo el dataset como valor por defecto.
riesgo_general = df['probabilidad_alto'].mean()
contratistas_con_datos = ['Contratista A', 'Contratista B', 'Contratista C']

riesgo_contratista = {}
for contratista in resumen.index:
    if contratista in contratistas_con_datos:
        riesgo_contratista[contratista] = float(resumen.loc[contratista, 'riesgo_promedio'])
    else:
        riesgo_contratista[contratista] = round(float(riesgo_general), 3)

print(resumen)
print(f"\nRiesgo general (valor por defecto para D y E): {riesgo_general:.3f}")
print("\nRiesgo asignado a cada contratista:")
for contratista, riesgo in riesgo_contratista.items():
    print(f"  {contratista}: {riesgo:.3f}")

               observaciones  riesgo_promedio  porcentaje_sobre_umbral
contratista                                                           
Contratista A             68            0.500                   60.294
Contratista B             24            0.524                   66.667
Contratista C             11            0.406                   54.545
Contratista D              1            0.929                  100.000
Contratista E              1            0.769                  100.000

Riesgo general (valor por defecto para D y E): 0.502

Riesgo asignado a cada contratista:
  Contratista A: 0.500
  Contratista B: 0.524
  Contratista C: 0.406
  Contratista D: 0.502
  Contratista E: 0.502


In [13]:
# --- Paso 9: Grafo de zonas y búsqueda de costo uniforme (NA)---


# --- Paso 9: Grafo de frentes de trabajo y búsqueda de costo uniforme ---
import heapq

# Nodos: frente de trabajo de cada actividad y el contratista que la ejecuta.
# Los tiempos de traslado son estimaciones aproximadas basadas en el
# recorrido habitual de inspección en sitio.
zona_contratista = {
    'Acceso':                  None,             # punto de partida
    'High racking':            'Contratista A',
    'Etiquetas y accesorios':  'Contratista C',
    'Sistema eléctrico':       'Contratista B',
    'Sistema contra incendio': 'Contratista B',
    'HVAC':                    'Contratista B',
    'Oficina HSE':             None,             # punto de llegada
}

# Tiempos de traslado en minutos entre frentes vecinos (en ambos sentidos)
traslados = [
    ('Acceso', 'High racking', 3),
    ('Acceso', 'Sistema eléctrico', 4),
    ('High racking', 'Etiquetas y accesorios', 2),
    ('High racking', 'Sistema contra incendio', 3),
    ('Etiquetas y accesorios', 'HVAC', 4),
    ('Etiquetas y accesorios', 'Oficina HSE', 6),
    ('Sistema eléctrico', 'HVAC', 5),
    ('Sistema contra incendio', 'HVAC', 3),
    ('Sistema contra incendio', 'Oficina HSE', 5),
    ('HVAC', 'Oficina HSE', 4),
]

def riesgo_zona(zona):
    contratista = zona_contratista[zona]
    return riesgo_contratista.get(contratista, 0) if contratista else 0

# Dos grafos: uno con tiempo real y otro con costo ponderado por riesgo
grafo_tiempo = {zona: [] for zona in zona_contratista}
grafo_ponderado = {zona: [] for zona in zona_contratista}
for a, b, minutos in traslados:
    grafo_tiempo[a].append((b, minutos))
    grafo_tiempo[b].append((a, minutos))
    grafo_ponderado[a].append((b, minutos * (1 - riesgo_zona(b))))
    grafo_ponderado[b].append((a, minutos * (1 - riesgo_zona(a))))

def busqueda_costo_uniforme(grafo, inicio, meta):
    frontera = [(0, inicio, [inicio])]
    explorados = set()
    while frontera:
        costo, nodo, camino = heapq.heappop(frontera)
        if nodo == meta:
            return camino, costo
        if nodo in explorados:
            continue
        explorados.add(nodo)
        for vecino, costo_arista in grafo[nodo]:
            if vecino not in explorados:
                heapq.heappush(frontera, (costo + costo_arista, vecino, camino + [vecino]))
    return None, float('inf')

def describir_ruta(nombre, camino):
    tiempos = {(a, b): m for a, b, m in traslados}
    tiempos.update({(b, a): m for a, b, m in traslados})
    tiempo_total = sum(tiempos[(a, b)] for a, b in zip(camino, camino[1:]))
    riesgo_cubierto = sum(riesgo_zona(zona) for zona in camino)
    print(f"{nombre}:")
    print(f"  Recorrido: {' → '.join(camino)}")
    print(f"  Tiempo real: {tiempo_total} minutos")
    print(f"  Riesgo acumulado de los frentes visitados: {riesgo_cubierto:.3f}\n")

ruta_rapida, _ = busqueda_costo_uniforme(grafo_tiempo, 'Acceso', 'Oficina HSE')
ruta_riesgo, _ = busqueda_costo_uniforme(grafo_ponderado, 'Acceso', 'Oficina HSE')

describir_ruta("Ruta ingenua (solo tiempo)", ruta_rapida)
describir_ruta("Ruta ponderada por riesgo", ruta_riesgo)

Ruta ingenua (solo tiempo):
  Recorrido: Acceso → High racking → Etiquetas y accesorios → Oficina HSE
  Tiempo real: 11 minutos
  Riesgo acumulado de los frentes visitados: 0.906

Ruta ponderada por riesgo:
  Recorrido: Acceso → High racking → Sistema contra incendio → Oficina HSE
  Tiempo real: 11 minutos
  Riesgo acumulado de los frentes visitados: 1.024




## Riesgo por contratista (Paso 8)

El riesgo de cada contratista es el promedio de las probabilidades que el perceptrón asignó a sus observaciones. D y E, con una sola observación cada uno, reciben el promedio general (0.502) como valor por defecto.

| Contratista | Observaciones | Riesgo asignado |
|---|---|---|
| A | 68 | 0.500 |
| B | 24 | 0.524 |
| C | 11 | 0.406 |
| D | 1 | 0.502 (por defecto) |
| E | 1 | 0.502 (por defecto) |

## Rutas de inspección (Paso 9)

Los nodos son frentes de trabajo por actividad, asociados a su contratista. Los tiempos de traslado son estimaciones basadas en el recorrido habitual de inspección. El costo ponderado de llegar a un frente es `minutos × (1 − riesgo del frente)`.

| Ruta | Recorrido | Tiempo real | Riesgo acumulado |
|---|---|---|---|
| Ingenua (solo tiempo) | Acceso → High racking → Etiquetas y accesorios → Oficina HSE | 11 min | 0.906 |
| Ponderada por riesgo | Acceso → High racking → Sistema contra incendio → Oficina HSE | 11 min | 1.024 |

**Resultado:** con el mismo tiempo de recorrido, el agente sustituye el frente de etiquetas (riesgo 0.406) por el de sistema contra incendio (riesgo 0.524) y aumenta el riesgo cubierto por la inspección.

**Limitación principal:** el riesgo se calcula por contratista y no por frente de trabajo, y las diferencias entre contratistas son pequeñas. La documentación completa está en `metodologia.md`.